In [ ]:
import torch.nn as nn
import pandas as pd

In [6]:
data = pd.read_csv("data/Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


## **Descripción de la tarea**

Trabajaremos con un dataset de la serie "Pokémon" que desglosa las capacidades de cada criatura en atributos cuantitativos. La tarea consiste en utilizar arquitecturas de redes neuronales simples (MLP) para identificar patrones que distingan a los Pokémon Legendarios del resto. Deberán procesar estas variables y entrenar un clasificador que maximice la capacidad predictiva sobre la variable objetivo "Legendary".

Consideraciones:
- Debe entregarlo a más tardar el 29 de mayo a las 18:00 horas.
- Debe ser entregado al correo luis.llanca@uach.cl con el asunto "Tarea-1-MLP", el archivo debe llamarse NG-MLP-Tarea1.ipynb donde NG es el número de grupo. Es importante que el asunto sea exactamente el mismo. También, se les pedirá que se anoten en la plantilla (que se compartirá posteriormente) para una pequeña interrogación.
- Por cada 20 minutos de retraso, se descontará una décima de la nota. 
- Si necesitan ayuda, pueden escribir a los correos luis.llanca@uach.cl, luis.llanca@cenia.cl o escribir al discord de usuario: llanking (tengo una foto de mi gata). PD: prefiero mucho más el discord. 

### **Explicación del dataset**
Explique el dataset en detalle, incluyendo como mínimo una pequeña descripción de cada columna, el tipo de datos que contiene cada columna, y cualquier información adicional relevante para entender el dataset.

El dataset contiene información de los Pokémon de primera a sexta generación, incluyendo sus estadísticas, tipos elementales, generación y una variable indicadora de si son o no legendarios. También incluye un identificador único de cada Pokémon, el cual puede repetirse para distintas versiones del mismo, como megaevoluciones, diferentes formas (como en el caso de Tornadus), tamaños, etc.

#### **Descripción de las columnas:**

| Columna    | Tipo de dato | Descripción                                                                                                       |
| :--------- | :----------- | :---------------------------------------------------------------------------------------------------------------- |
| #          | Numérica     | Identificador único para distinguir los Pokémon; se repite si un mismo Pokémon tiene múltiples formas o versiones |
| Name       | String       | Nombre del Pokémon o de su versión específica                                                                     |
| Type 1     | String       | Tipo elemental primario del Pokémon                                                                               |
| Type 2     | String o NaN | Tipo elemental secundario del Pokémon                                                                             |
| Total      | Numérica     | Suma total de las estadísticas                                                                                    |
| HP         | Numérica     | Puntos de vida del Pokémon                                                                                        |
| Attack     | Numérica     | Puntos de ataque del Pokémon                                                                                      |
| Defense    | Numérica     | Puntos de defensa del Pokémon                                                                                     |
| Sp. Atk    | Numérica     | Puntos de ataque especial del Pokémon                                                                             |
| Sp. Def    | Numérica     | Puntos de defensa especial del Pokémon                                                                            |
| Speed      | Numérica     | Puntos de velocidad del Pokémon                                                                                   |
| Generation | Numérica     | Variable categórica numérica que indica la generación del Pokémon                                                 |
| Legendary  | Booleana     | Variable indicadora que señala si el Pokémon es legendario                                                        |

#### **Consideraciones**

Se debe tener en cuenta, al analizar el dataset, la existencia del *power creep* en Pokémon, donde los Pokémon de generaciones más recientes tienden a ser más fuertes que los de generaciones anteriores. Esto podría generar que algunos Pokémon de primera generación no sean identificados como legendarios por el modelo.

También es importante considerar que el dataset está desbalanceado por naturaleza, ya que existen aproximadamente 60 Pokémon legendarios, mientras que el resto corresponde a Pokémon no legendarios.

Finalmente, se debe tener en cuenta la presencia de los pseudo-legendarios, que son Pokémon cuyas estadísticas totales superan los 600 puntos, pero que no son clasificados como legendarios.

### **Preparación del dataset**

Realice una preparación del dataset según lo que considere necesario para el entrenamiento de un modelo de clasificación. Justifique las decisiones que tome en este proceso.

In [27]:
# Primer filtro, quitar las distintas versiones de un mismo pokemon.
# Esto quita duplicados guiandose por la columna #, y se queda con la primera fila en casos de duplicados (Porque notamos que el dataset venia ordenado, osea las versiones de Charizar estaban debajo de el pokemon Charizard)
# # | Pokemon
# 6 | Charizard
# 6 | CharizardMega Charizard X
# 6 | CharizardMega Charizard Y
# Entonces nos quedamos con el primero simplemente agrupando por #
df_clean = data.drop_duplicates(subset="#", keep="first")

# Segundo filtro, aplicar el one hot encoding a las variables categoricas (Type 1, Type2 y Generacion)
# Se rellenan los NaN con None, para aplicar los dummies
df_clean["Type 2"] = df_clean["Type 2"].fillna("None")
# Se aplica el get dummies, que pasa las posibles valores de la columna a columnas tipo flag
df_encoded = pd.get_dummies(
    df_clean,
    columns=["Type 1", "Type 2", "Generation"],
    prefix=["T1", "T2", "Gen"]
)

# Tercer filtro, Quitar columnas Total, ya que es redundante, # y Name ya que no nos aportan informacion relevante
df_final = df_encoded.drop(columns=["Total", "#", "Name"])

In [29]:
df_final.head(5)

,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Legendary,T1_Bug,T1_Dark,T1_Dragon,...,T2_Psychic,T2_Rock,T2_Steel,T2_Water,Gen_1,Gen_2,Gen_3,Gen_4,Gen_5,Gen_6
0,45,49,49,65,65,45,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
1,60,62,63,80,80,60,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,80,82,83,100,100,80,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
4,39,52,43,60,50,65,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
5,58,64,58,80,65,80,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False


### Definición del modelo  
Defina al menos 3 arquitecturas de redes neuronales simples (MLP) para el problema de clasificación. Justifique las decisiones que tome en la definición de cada arquitectura. Las definiciones se deben hacer en un archivo ```models.py``` e importarlas en este cuadernillo. Debe seleccionar "la mejor" arquitectura para el entrenamiento, y justificar su elección.

### Definición de optimizador y función de costo  
Defina un optimizador y una función de costo adecuado para el entrenamiento del modelo. Justifique sus decisiones.

### Entrenamiento del modelo
Entrene el modelo seleccionado utilizando el dataset preparado. Justifique las decisiones que tome en el proceso de entrenamiento, incluyendo la selección de hiperparámetros, el número de épocas, el tamaño del batch, etc.

### Evaluación del modelo
Evalúe el modelo utilizando métricas adecuadas para este problema de clasificación. Justifique la selección de las métricas utilizadas y discuta los resultados obtenidos. 

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?



### IA Generativa

Con el fin de ocupar IA Generativa de manera responsable, se les pide que respondan a las siguientes preguntas:
1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.
2. ¿En qué parte o partes de la tarea utilizó estas herramientas?